# IQM: Bell z losowaniem ustawień i mitygacją

Stan dwóch qutritów w bazie kanonicznej, przygotowany przez `CZ3(F3 ⊗ F3)|00⟩`.
Używamy przypiętych artefaktów z 2026-09-09: F3 — 2 CX, CZ3 — 6 CZ.
To najlepsza wskazana tutaj dostępna synteza, bez dowodu globalnej optymalności CZ3.
Przy wczytaniu sprawdzamy błąd bramek na całej podprzestrzeni kodowej, leakage,
SHA-256 i zgodność stanu z referencją Bella.

**Run All jest offline. Nie łączy się z IQM i nie wysyła jobów.**

Porównujemy:
1. **RAW**: DD wyłączone, bez twirlingu, bez korekcji odczytu, bez ZNE.
2. **MM**: korekcja odczytu na tych samych danych RAW.
3. **MM_TWIRL_DD**: nowe pomiary, Pauli twirling natywnych CZ i natywne DD.
4. **MM_TWIRL_DD_ZNE**: liniowa ekstrapolacja wariantu 3 dla skal CZ 1, 3, 5.

Każdy blok losuje niezależnie ustawienie każdej strony (`secrets`). Losowania
trwają do minimum bloków i pokrycia wymaganych korelatorów, z twardym limitem.
Jeden zapisany harmonogram jest wspólny dla wariantów. Powtórzone ustawienia
zachowują osobne identyfikatory bloków. Jest to eksperyment blokowy, nie nowe
losowanie ustawienia przed każdym pojedynczym shotem. System RNG nie jest QRNG.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src/qudits_on_qubits').is_dir())
for path in (ROOT, ROOT / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from scripts.iqm_randomized_bell_campaign import (
    load_state, prepare_campaign, execute_campaign, analyze_campaign, write_json,
)
from qudits_on_qubits.experiments import RandomizedBlocks, IQMHardware, TranspilationConfig
from qudits_on_qubits.experiments.setting_schedule import generate_schedule, SettingSchedule
from qudits_on_qubits.experiments.preparation import prepare_measurements
from qudits_on_qubits.experiments.block_estimation import evaluate_blocks
from qudits_on_qubits.reference_experiments import get_reference_experiment


In [ ]:
RUN_HARDWARE = False
RUN_RECOVERY = False
DEVICE = 'emerald'  # Ta wersja wymaga architektury natywnej r/CZ bez MOVE.
SETTING_DRAWS = 100
SHOTS_PER_DRAW = 1024
MAX_SETTING_DRAWS = 1024
TWIRLING_INSTANCES = 8  # shots_per_draw / instances = 128 shoty na wariant
TWIRLING_SEED = 20260909
CALIBRATION_SHOTS = 4096
MAX_CIRCUITS_PER_JOB = 100
TRANSPILE = TranspilationConfig(optimization_level=3, seed_transpiler=9)
GATE_DIRECTORY = ROOT / 'experiment_inputs/iqm_randomized_bell/canonical_optimized_20260909'
WORK = ROOT / 'artifacts/iqm_randomized_bell'
WORK.mkdir(parents=True, exist_ok=True)
SCHEDULE_PATH = WORK / 'schedule_emerald_1024shots_100settings.json'
# Wskaż istniejącą kampanię do analizy; None nie uruchamia sprzętu.
SAVED_CAMPAIGN = None


In [ ]:
artifacts, gate_evidence = load_state(GATE_DIRECTORY)
display(pd.DataFrame({k: v for k, v in gate_evidence.items() if isinstance(v, dict)}).T)
print('Fidelity przygotowanego stanu:', gate_evidence['state_fidelity'])
reference = get_reference_experiment('two_qutrit')
measurement = RandomizedBlocks(SETTING_DRAWS, SHOTS_PER_DRAW, MAX_SETTING_DRAWS,
                               MAX_CIRCUITS_PER_JOB)
if SCHEDULE_PATH.exists():
    schedule = SettingSchedule.from_safe_dict(json.loads(SCHEDULE_PATH.read_text()))
    if schedule.config != measurement:
        raise ValueError('Zmieniono konfigurację. Wskaż nowy SCHEDULE_PATH; nie nadpisuj starego losowania.')
else:
    schedule = generate_schedule(reference, measurement)
    if not schedule.complete:
        raise ValueError('Niepełne pokrycie ustawień. Zwiększ MAX_SETTING_DRAWS.')
    write_json(SCHEDULE_PATH, schedule.to_safe_dict())
prepared = prepare_measurements(artifacts, schedule)
display(pd.DataFrame([b.to_safe_dict() for b in schedule.blocks]))


## Kontrola lokalna

Poniższa symulacja bezszumowa sprawdza konwencję bitów i estymator blokowy.
Nie emuluje DD ani skuteczności mitygacji na urządzeniu. Raport RAW zawiera
wartość bez postselekcji, leakage i przedziały ufności istniejącego pipeline’u.


In [ ]:
from qiskit.quantum_info import Statevector
ideal_counts = {}
catalog = prepared.metadata['catalog_index_by_block_id']
for block in schedule.blocks:
    circuit = prepared.circuits[catalog[block.block_id]].remove_final_measurements(inplace=False)
    state = Statevector(circuit)
    state.seed(1000 + block.block_id)
    ideal_counts[block.block_id] = {str(k): int(v) for k, v in state.sample_counts(SHOTS_PER_DRAW).items()}
ideal_report = evaluate_blocks(reference, schedule, ideal_counts,
    prepared.metadata['qutrit_bit_indices_by_setting'], prepared.metadata['encoding_outcome_map'])
display(ideal_report)


## Offline preflight IQM i budżet

Kompilacja na `IQMFakeGarnet`, bez połączenia. Na prawdziwym backendzie obwody
będą kompilowane ponownie według jego kalibracji. Wszystkie ramiona korzystają
z tego samego skompilowanego katalogu; pomiarowa mapa fizyczna jest zapisywana.

Measurement mitigation: pełna macierz przypisań **16 × 16** dla każdej użytej
czwórki fizycznych kubitów, ze wszystkimi stanami bazowymi. Korekcja `A⁻¹p`
zachowuje ujemne quasi-prawdopodobieństwa. Macierz źle uwarunkowana przerywa
analizę. Błędy przygotowania stanów kalibracyjnych są częścią ograniczenia metody.

Twirling: niezależne pary Pauliego przed i po każdym fizycznym CZ. Budżet shotów
na blok jest dzielony między realizacje; nie mnożymy go przez ich liczbę.
ZNE: folding fizycznych CZ po kompilacji, potem twirling, bez ponownej optymalizacji.
Skale 1/3/5 dotyczą liczby CZ, nie całkowitego szumu (w tym DD, odczytu i bramek 1q).
DD wykonuje kompilator pulsowy IQM przez `STANDARD_DD_STRATEGY`; preflight nie
potwierdza liczby wstawionych pulsów ani poprawy jakości.
[API DD](https://docs.iqm.tech/iqm-client/api/iqm.iqm_client.models.CircuitCompilationOptions.html).


In [ ]:
from iqm.qiskit_iqm.fake_backends.fake_garnet import IQMFakeGarnet
from qudits_on_qubits.experiments.backends.iqm import IQMAdapter
fake_adapter = IQMAdapter(IQMHardware(device=DEVICE), backend=IQMFakeGarnet())
_, batches, calibrations = prepare_campaign(artifacts, schedule, fake_adapter,
    TRANSPILE, TWIRLING_INSTANCES, TWIRLING_SEED)
budget = pd.DataFrame([{
    'variant': b['arm'], 'factor': b['factor'], 'circuits': len(b['circuits']),
    'shots_per_circuit': b['shots'], 'total_shots': len(b['circuits']) * b['shots'],
    'CZ_min': min(c.count_ops().get('cz', 0) for c in b['circuits']),
    'CZ_max': max(c.count_ops().get('cz', 0) for c in b['circuits']),
} for b in batches])
display(budget)
print('Pomiary:', int(budget.total_shots.sum()))
print('Kalibracja:', 16 * len(calibrations) * CALIBRATION_SHOTS, 'shotów;', len(calibrations), 'map fizycznych')
print('MM nie wymaga nowych pomiarów Bella. Skala 1 jest wspólna z ZNE.')


## Wykonanie na hardware — domyślnie zablokowane

Nie uruchomiono hardware przy tworzeniu notatnika. Poniższa komórka zadziała
dopiero po ręcznym ustawieniu `RUN_HARDWARE = True`. Używa istniejącego loadera
IQM i konfiguracji środowiskowej; sekretów nie zapisuje w notatniku.
Flaga resetuje się przed połączeniem. Ponowne wykonanie komórki nie powtarza jobów.
Nowa kampania wymaga nowej flagi i tworzy nowy katalog.

Najpierw kalibracje odczytu, potem RAW oraz skale 1/3/5 z twirlingiem i DD.
Taki porządek nie usuwa dryfu w czasie; zapisz kalibrację backendu i wnioskuj
ostrożnie z porównania. W każdym pliku wynikowym zachowane są bloki i realizacje.


In [ ]:
if RUN_HARDWARE:
    RUN_HARDWARE = False
    from uuid import uuid4
    adapter = IQMAdapter(IQMHardware(device=DEVICE))
    real_prepared, real_batches, real_calibrations = prepare_campaign(
        artifacts, schedule, adapter, TRANSPILE, TWIRLING_INSTANCES, TWIRLING_SEED)
    print('Rzeczywisty budżet:', sum(len(b['circuits']) * b['shots'] for b in real_batches)
          + 16 * len(real_calibrations) * CALIBRATION_SHOTS)
    SAVED_CAMPAIGN = WORK / ('campaign_' + uuid4().hex)
    execute_campaign(SAVED_CAMPAIGN, schedule, gate_evidence, real_batches,
        real_calibrations, adapter.backend, allow_hardware=True,
        calibration_shots=CALIBRATION_SHOTS, max_circuits=MAX_CIRCUITS_PER_JOB)
else:
    print('Hardware wyłączony. Nie wysłano jobów.')


## Odczyt po przerwaniu i analiza offline

Jeśli wykonanie przerwano po zapisaniu job ID, można pobrać wynik bez ponownego
submitu. `submission_unknown` bez ID wymaga sprawdzenia dashboardu IQM;
nie wolno automatycznie ponawiać takiego żądania. Komórka recovery tylko pobiera
istniejące joby. Nie uruchamia brakujących partii; niepełna kampania nie daje
pełnej analizy. Dokończenie niewysłanych partii wymaga osobnej decyzji.


In [ ]:
if RUN_RECOVERY:
    RUN_RECOVERY = False
    if SAVED_CAMPAIGN is None:
        raise ValueError('Wskaż SAVED_CAMPAIGN.')
    adapter = IQMAdapter(IQMHardware(device=DEVICE))
    for name in json.loads((Path(SAVED_CAMPAIGN) / 'index.json').read_text()):
        output = Path(SAVED_CAMPAIGN) / f'{name}.counts.json'
        if output.exists():
            continue
        marker = Path(SAVED_CAMPAIGN) / f'{name}.submission.json'
        if not marker.exists():
            print(name, 'nie wysłano — bez automatycznego submitu')
            continue
        entry = json.loads(marker.read_text())
        if 'job_id' not in entry:
            raise RuntimeError(f'{name}: submission_unknown; sprawdź dashboard IQM')
        request = json.loads((Path(SAVED_CAMPAIGN) / f'{name}.request.json').read_text())
        job = adapter.backend.retrieve_job(entry['job_id'])
        result = job.result()
        size = 16 if 'mapping' in request else len(request['records'])
        write_json(output, [result.get_counts(i) for i in range(size)])


In [ ]:
if SAVED_CAMPAIGN is not None:
    saved_schedule = SettingSchedule.from_safe_dict(json.loads(
        (Path(SAVED_CAMPAIGN) / 'schedule.json').read_text()))
    saved_prepared = prepare_measurements(artifacts, saved_schedule)
    analysis = analyze_campaign(SAVED_CAMPAIGN, saved_schedule, saved_prepared.metadata)
    table = pd.DataFrame(analysis['rows'])
    display(table)
    display(analysis['raw_block_analysis'])
    table.plot.bar(x='variant', y='real', title='Bell: część rzeczywista (skale ZNE pokazane osobno)')
else:
    print('Brak kampanii hardware do analizy.')


### Interpretacja

Główny estymator używa wszystkich shotów: leakage wnosi zero, bez renormalizacji
na pozostałych wynikach. RAW ma dodatkowo raport blokowy istniejącego pipeline’u
z leakage i wariantem warunkowym. Wynik warunkowy nie zastępuje RAW.

Wartości MM oraz ZNE są estymatorami modelowymi, nie surowymi częstościami.
Nie obcinamy wyniku ZNE ani ujemnych wag do zakresu fizycznego. Raportujemy obie
części wartości zespolonej i reszty dopasowania ZNE. Nie przypisujemy mitygowanym
wynikom przedziałów Hoeffdinga z RAW; ten notatnik nie wyznacza ich niepewności.
Wynik przekraczający granicę Bella po mitygacji sam nie stanowi certyfikatu
naruszenia bez dodatkowych założeń. Wspólny harmonogram służy porównaniu technik;
to nie jest test zamykający luki lokalności i niezależności wyboru ustawień.
